# 03 — Feature Engineering

**Projeto:** Financial Behavior Intelligence  
**Objetivo:** Construir um DataFrame agregado por usuário com features que descrevem o comportamento financeiro. Este arquivo será o input do modelo de ML.

---

## 0. Setup

In [129]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/transactions_clean.csv')

# convertendo a data para garantir que funcione
df['Date'] = pd.to_datetime(df['Date'])

print('Shape:', df.shape)
print("Agora sim! Dados carregados ✅")
df.head(3)

Shape: (19046, 11)
Agora sim! Dados carregados ✅


,User_ID,Date,Description,Amount,Transaction Type,Category,Account Name,Year,Month,Month_Name,Period
0,USER_0001,2018-01-01,Amazon,11.11,debit,Shopping,credit card,2018,1,Jan,2018-01
1,USER_0001,2018-01-02,Mortgage Payment,1247.44,debit,Mortgage & Rent,checking,2018,1,Jan,2018-01
2,USER_0001,2018-01-02,Thai Restaurant,24.22,debit,Restaurants,credit card,2018,1,Jan,2018-01


## 1. Agregações Mensais por Usuário

In [130]:
# criando o resumo mensal básico por usuário

features_basicas = df.groupby(['User_ID', 'Period']).agg({
    'Amount': ['sum', 'mean', 'count']
}).reset_index()

# ajustando nomes para ficar organizado
features_basicas.columns = ['User_ID', 'Period', 'Total_Gasto', 'Media_Transacao', 'Qtd_Transacoes']

# criando a visão de Crédito vs Débito 
pivot_cred_deb = df.pivot_table(
    index=['User_ID', 'Period'],
    columns='Transaction Type',
    values='Amount',
    aggfunc='sum',
    fill_value=0
).reset_index()

# mesclando as duas tabelas
features_completas = pd.merge(features_basicas, pivot_cred_deb, on=['User_ID', 'Period'])

# criando a coluna de saldo final
features_completas['Saldo_Mensal'] = features_completas.get('credit', 0) - features_completas.get('debit', 0)

print(f"Shape: {features_completas.shape}")

print("Estrutura de features criada 📊 ")
features_completas.head()

Shape: (1221, 8)
Estrutura de features criada 📊 


,User_ID,Period,Total_Gasto,Media_Transacao,Qtd_Transacoes,credit,debit,Saldo_Mensal
0,USER_0001,2018-01,10094.34,315.448125,32,7162.89,2931.45,4231.44
1,USER_0001,2018-02,8385.80,254.115152,33,5220.75,3165.05,2055.70
2,USER_0001,2018-03,10821.66,251.666512,43,7321.50,3500.16,3821.34
3,USER_0001,2018-04,13196.42,274.925417,48,7166.88,6029.54,1137.34
4,USER_0001,2018-05,16483.58,412.089500,40,5091.55,11392.03,-6300.48


In [131]:
# saldo mensal real (receita - despesa)
# total recebido (crédito) por usuário + período
# Credit Card Payment é pagamento de fatura, não receita real, entao será removido
    
receita_mensal = df[
    (df['Transaction Type'] == 'credit') &
    (df['Description'] != 'Credit Card Payment')
].groupby(['User_ID', 'Period'])['Amount'].sum().reset_index()
receita_mensal.columns = ['User_ID', 'Period', 'Total_Receita']

# juntando com features_completas
features_completas = features_completas.merge(
    receita_mensal,
    on=['User_ID', 'Period'],
    how='left'
).fillna({'Total_Receita': 0})

# calculando o saldo
features_completas['Saldo_Mensal_Real'] = (
    features_completas['Total_Receita'] - features_completas['Total_Gasto']
).round(2)

print("Saldo mensal real criado!✅")
features_completas[['User_ID', 'Period', 'Total_Receita', 'Total_Gasto', 'Saldo_Mensal_Real']].head()

Saldo mensal real criado!✅


,User_ID,Period,Total_Receita,Total_Gasto,Saldo_Mensal_Real
0,USER_0001,2018-01,4000.0,10094.34,-6094.34
1,USER_0001,2018-02,4000.0,8385.80,-4385.80
2,USER_0001,2018-03,6000.0,10821.66,-4821.66
3,USER_0001,2018-04,4000.0,13196.42,-9196.42
4,USER_0001,2018-05,4000.0,16483.58,-12483.58


## 2. Features por Usuário

In [132]:
# --- Features de saldo e receita/despesa ---

monthly = features_completas[['User_ID', 'Period', 'debit', 'credit', 'Saldo_Mensal_Real']].copy()

feat_saldo = monthly.groupby('User_ID').agg(
    avg_monthly_debit   = ('debit',        'mean'), # média de quanto sai
    avg_monthly_credit  = ('credit',       'mean'), # média de quanto entra
    avg_saldo           = ('Saldo_Mensal_Real', 'mean'), # lucro médio do usuário
    std_saldo           = ('Saldo_Mensal_Real', 'std'),  # desvio padrão
    spending_volatility = ('debit',        'std'),  # grau de flutuação
    pct_meses_negativo  = ('Saldo_Mensal_Real', lambda x: round((x < 0).mean(), 3)), # qtd de meses com saldo negativo
    total_meses         = ('Saldo_Mensal_Real', 'count')
).reset_index()

# taxa de poupança média = (receita - despesa) / receita
feat_saldo['savings_rate'] = ( # quanto o usuário consegue guardar
    (feat_saldo['avg_monthly_credit'] - feat_saldo['avg_monthly_debit']) /
    feat_saldo['avg_monthly_credit'].replace(0, np.nan)
).round(3)

print(f'feat_saldo OK: {feat_saldo.shape[0]} usuarios x {feat_saldo.shape[1]-1} features')
feat_saldo.head()


feat_saldo OK: 51 usuarios x 8 features


,User_ID,avg_monthly_debit,avg_monthly_credit,avg_saldo,std_saldo,spending_volatility,pct_meses_negativo,total_meses,savings_rate
0,USER_0001,4575.418095,5917.607619,-6028.740000,2558.892979,2576.685869,1.0,21,0.227
1,USER_0002,3092.826667,4136.631250,-3092.826667,636.790311,636.790311,1.0,24,0.252
2,USER_0003,3367.328333,3124.895000,-3367.328333,687.632079,687.632079,1.0,24,-0.078
3,USER_0004,2943.169583,3349.859583,-2943.169583,694.305041,694.305041,1.0,24,0.121
4,USER_0005,2737.333333,4842.969167,-2737.333333,661.454993,661.454993,1.0,24,0.435


In [133]:
# --- Feature da variação de despesas mês a mês ---

features_completas = features_completas.sort_values(['User_ID', 'Period'])

features_completas['Variacao_Gasto_Mensal'] = (
    features_completas
    .groupby('User_ID')['Total_Gasto']  # agrupa por usuário
    .pct_change()                        # calcula a variação % em relação à linha anterior
    * 100                                # transforma em porcentagem
).round(2)

print("Variação mensal criada!✅")
features_completas[['User_ID', 'Period', 'Total_Gasto', 'Variacao_Gasto_Mensal']].head(10)
# Nota: o primeiro mês de cada usuário sempre sera NaN, tendo em vista que não há mês anterior pata comparação

Variação mensal criada!✅


,User_ID,Period,Total_Gasto,Variacao_Gasto_Mensal
0,USER_0001,2018-01,10094.34,NaN
1,USER_0001,2018-02,8385.80,-16.93
2,USER_0001,2018-03,10821.66,29.05
3,USER_0001,2018-04,13196.42,21.94
4,USER_0001,2018-05,16483.58,24.91
5,USER_0001,2018-06,9683.07,-41.26
6,USER_0001,2018-07,7635.32,-21.15
7,USER_0001,2018-08,9775.33,28.03
8,USER_0001,2018-09,8521.70,-12.82
9,USER_0001,2018-10,7870.58,-7.64


In [134]:
# --- Features de principais categorias (apenas débitos) ---

debits = df[df['Transaction Type'] == 'debit']

total_by_user = debits.groupby('User_ID')['Amount'].sum().rename('total_debit')

# % do gasto na principal categoria, onde o usuário mais gasta
top_cat = (debits.groupby(['User_ID', 'Category'])['Amount'].sum()
           .reset_index()
           .sort_values('Amount', ascending=False)
           .groupby('User_ID')
           .first()
           .reset_index()
           .rename(columns={'Category': 'top_category', 'Amount': 'top_category_spend'}))

top_cat = top_cat.merge(total_by_user, on='User_ID')
top_cat['top_category_pct'] = (top_cat['top_category_spend'] / top_cat['total_debit']).round(3)

# definição de categorias fixas (recorrentes)
FIXED_CATS = ['Rent', 'Phone Bill', 'Internet Bill', 'Insurance', 'Utilities']
fixed = (debits[debits['Category'].isin(FIXED_CATS)]
         .groupby('User_ID')['Category']
         .nunique()
         .rename('num_fixed_expenses')
         .reset_index())

# tem investimento?
has_inv = (debits[debits['Category'] == 'Investment']
           .groupby('User_ID')['Amount']
           .sum()
           .rename('total_investment')
           .reset_index())
has_inv['has_investment'] = 1

print('Features de categoria OK')

perfil_usuario = top_cat.merge(fixed, on='User_ID', how='left').merge(has_inv, on='User_ID', how='left').fillna(0)

perfil_usuario.head(10)



Features de categoria OK


,User_ID,top_category,top_category_spend,total_debit,top_category_pct,num_fixed_expenses,total_investment,has_investment
0,USER_0001,Credit Card Payment,33041.36,96083.78,0.344,1,0.00,0.0
1,USER_0002,Rent,31977.54,74227.84,0.431,5,0.00,0.0
2,USER_0003,Rent,31397.66,80815.88,0.389,5,0.00,0.0
3,USER_0004,Rent,31144.12,70636.07,0.441,5,0.00,0.0
4,USER_0005,Rent,20816.93,65696.00,0.317,5,5516.94,1.0
5,USER_0006,Rent,22584.41,54499.36,0.414,5,4252.04,1.0
6,USER_0007,Rent,44946.75,164566.27,0.273,5,0.00,0.0
7,USER_0008,Rent,32457.88,75769.25,0.428,5,0.00,0.0
8,USER_0009,Rent,21817.59,73998.86,0.295,5,5996.80,1.0
9,USER_0010,Rent,31579.04,76657.14,0.412,5,0.00,0.0


In [135]:
# --- Features de todas as categorias (apenas débitos) ---

df_gastos = df[df['Transaction Type'] == 'debit'].copy()

# somando gasto por usuário + período + categoria
gasto_por_categoria = df_gastos.groupby(
    ['User_ID', 'Period', 'Category']
)['Amount'].sum().reset_index()

# pivot - cada categoria vira uma coluna
gasto_categorias_wide = gasto_por_categoria.pivot_table(
    index=['User_ID', 'Period'],
    columns='Category',
    values='Amount',
    aggfunc='sum',
    fill_value=0
).reset_index()

# renomear colunas (padrão: gasto_nome_categoria)
gasto_categorias_wide.columns.name = None
novas_colunas = []
for col in gasto_categorias_wide.columns:
    if col in ['User_ID', 'Period']:
        novas_colunas.append(col)
    else:
        novo_nome = 'Gasto_' + col.replace(' ', '_').replace('&', '').replace('__', '_')
        novas_colunas.append(novo_nome)
gasto_categorias_wide.columns = novas_colunas

# juntando com as features básicas
features_completas = features_completas.merge(
    gasto_categorias_wide,
    on=['User_ID', 'Period'],
    how='left'
)

print(f"Features de todas as categoria criadas!✅") 
print(f"Shape final: {features_completas.shape}")

features_completas.head()



Features de todas as categoria criadas!✅
Shape final: (1221, 56)


,User_ID,Period,Total_Gasto,Media_Transacao,Qtd_Transacoes,credit,debit,Saldo_Mensal,Total_Receita,Saldo_Mensal_Real,...,Gasto_Restaurants,Gasto_Rideshare,Gasto_Savings_Transfer,Gasto_Shopping,Gasto_Software_Apps,Gasto_Streaming_Services,Gasto_Taxes,Gasto_Television,Gasto_Travel,Gasto_Utilities
0,USER_0001,2018-01,10094.34,315.448125,32,7162.89,2931.45,4231.44,4000.0,-6094.34,...,156.80,0.0,0.0,100.37,0.0,0.0,0.0,0.0,0.0,140.0
1,USER_0001,2018-02,8385.80,254.115152,33,5220.75,3165.05,2055.70,4000.0,-4385.80,...,290.85,0.0,0.0,11.11,0.0,0.0,0.0,0.0,0.0,160.0
2,USER_0001,2018-03,10821.66,251.666512,43,7321.50,3500.16,3821.34,6000.0,-4821.66,...,234.05,0.0,0.0,73.85,0.0,0.0,0.0,0.0,0.0,147.0
3,USER_0001,2018-04,13196.42,274.925417,48,7166.88,6029.54,1137.34,4000.0,-9196.42,...,202.27,0.0,0.0,54.47,0.0,0.0,0.0,0.0,0.0,125.0
4,USER_0001,2018-05,16483.58,412.089500,40,5091.55,11392.03,-6300.48,4000.0,-12483.58,...,127.30,0.0,0.0,219.45,0.0,0.0,0.0,0.0,0.0,125.0


In [136]:
# --- Uso de cartão de crédito ---

gasto_credito = df[
    (df['Transaction Type'] == 'debit') &   # só gastos
    (df['Account Name'].str.lower() == 'credit card')   # só os feitos no crédito
].groupby(['User_ID', 'Period'])['Amount'].sum().reset_index()

gasto_credito.columns = ['User_ID', 'Period', 'Total_Gasto_Credito']

# merge com features_completas (que já tem o total_gasto)
features_completas = features_completas.merge(
    gasto_credito,
    on=['User_ID', 'Period'],
    how='left'
)

features_completas['Total_Gasto_Credito'] = features_completas['Total_Gasto_Credito'].fillna(0)  # se não usou crédito naquele mês, é 0

# calculando a porcentagem: (quanto foi no crédito / total gasto) * 100
features_completas['Perc_Gasto_Credito'] = (
    (features_completas['Total_Gasto_Credito'] / features_completas['Total_Gasto']) * 100
).round(2)

print("Métrica de dependência de crédito criada! ✅")
features_completas[['User_ID', 'Period', 'Total_Gasto_Credito', 'Perc_Gasto_Credito']].head()

Métrica de dependência de crédito criada! ✅


,User_ID,Period,Total_Gasto_Credito,Perc_Gasto_Credito
0,USER_0001,2018-01,519.76,5.15
1,USER_0001,2018-02,517.49,6.17
2,USER_0001,2018-03,619.71,5.73
3,USER_0001,2018-04,1250.71,9.48
4,USER_0001,2018-05,873.95,5.30


In [137]:
# --- features mensais para nível de usuário ---
# features_completas está no nível mensal (1221 linhas), vamos resumir para 1 linha por usuário (51 linhas)

agg_monthly = features_completas.groupby('User_ID').agg(
    avg_variacao_gasto_mensal = ('Variacao_Gasto_Mensal', 'mean'),
    std_variacao_gasto_mensal = ('Variacao_Gasto_Mensal', 'std'),
    avg_perc_gasto_credito    = ('Perc_Gasto_Credito',    'mean'),
    avg_qtd_transacoes_mes    = ('Qtd_Transacoes',        'mean') 
).reset_index()

# Arredondamento e limpeza
cols_resumo = ['avg_variacao_gasto_mensal', 'std_variacao_gasto_mensal', 
               'avg_perc_gasto_credito', 'avg_qtd_transacoes_mes']

agg_monthly[cols_resumo] = agg_monthly[cols_resumo].round(3).fillna(0)

print(f'agg_monthly OK: {agg_monthly.shape[0]} usuários. ✅')
agg_monthly.head()

agg_monthly OK: 51 usuários. ✅


,User_ID,avg_variacao_gasto_mensal,std_variacao_gasto_mensal,avg_perc_gasto_credito,avg_qtd_transacoes_mes
0,USER_0001,4.564,27.203,6.419,38.381
1,USER_0002,15.841,85.475,8.355,15.000
2,USER_0003,22.214,90.324,8.022,15.000
3,USER_0004,2.596,24.019,8.094,15.000
4,USER_0005,18.305,113.616,4.007,16.000


In [138]:
# --- Dominância de categoria por mês ---

idx_dominante = df[df['Transaction Type'] == 'debit'].groupby(['User_ID', 'Period'])['Amount'].idxmax()
categoria_dominante = df.loc[idx_dominante, ['User_ID', 'Period', 'Category', 'Amount']]

categoria_dominante.columns = ['User_ID', 'Period', 'Categoria_Dominante', 'Valor_Dominante']

features_completas = features_completas.merge(
    categoria_dominante,
    on=['User_ID', 'Period'],
    how='left'
)

# criando métrica de "peso da dominante" (% do gasto total que essa categoria levou)
features_completas['Perc_Dominante'] = (
    (features_completas['Valor_Dominante'] / features_completas['Total_Gasto']) * 100
).round(2)

print("Feature de Categoria Dominante Mensal criada! 🏆")
features_completas[['User_ID', 'Period', 'Categoria_Dominante', 'Perc_Dominante']].head(10)

Feature de Categoria Dominante Mensal criada! 🏆


,User_ID,Period,Categoria_Dominante,Perc_Dominante
0,USER_0001,2018-01,Mortgage & Rent,12.36
1,USER_0001,2018-02,Mortgage & Rent,14.88
2,USER_0001,2018-03,Mortgage & Rent,11.53
3,USER_0001,2018-04,Mortgage & Rent,9.45
4,USER_0001,2018-05,Home Improvement,48.53
5,USER_0001,2018-06,Mortgage & Rent,12.88
6,USER_0001,2018-07,Mortgage & Rent,16.34
7,USER_0001,2018-08,Mortgage & Rent,12.76
8,USER_0001,2018-09,Mortgage & Rent,14.64
9,USER_0001,2018-10,Mortgage & Rent,15.36


## 3. Consolidar todas as features

In [139]:
# --- todas as features em 1 linha por usuário ---

user_features = feat_saldo.copy()

for nome, df_feat in [
    ('perfil_usuario', perfil_usuario),
    ('agg_monthly',    agg_monthly),
]:
    antes = user_features.shape[1]
    user_features = user_features.merge(df_feat, on='User_ID', how='left')
    print(f'  [{nome}] +{user_features.shape[1] - antes} colunas -> total: {user_features.shape[1]-1}')

print(f'\nShape final: {user_features.shape[0]} usuarios x {user_features.shape[1]-1} features')

  [perfil_usuario] +7 colunas -> total: 15
  [agg_monthly] +4 colunas -> total: 19

Shape final: 51 usuarios x 19 features


In [140]:
# --- última checagem de qualidade ---

nulls = user_features.isnull().sum()
print('Nulos por coluna:')
if nulls.sum() > 0:
    print(nulls[nulls > 0])
else:
    print('  Nenhum nulo encontrado. ✅')
print()
print('Colunas geradas:')
for col in user_features.columns[1:]:
    print(f'  {col}')

Nulos por coluna:
  Nenhum nulo encontrado. ✅

Colunas geradas:
  avg_monthly_debit
  avg_monthly_credit
  avg_saldo
  std_saldo
  spending_volatility
  pct_meses_negativo
  total_meses
  savings_rate
  top_category
  top_category_spend
  total_debit
  top_category_pct
  num_fixed_expenses
  total_investment
  has_investment
  avg_variacao_gasto_mensal
  std_variacao_gasto_mensal
  avg_perc_gasto_credito
  avg_qtd_transacoes_mes


In [ ]:
# --- preview ordenado por savings_rate ---

# mostrando o espectro do mais poupador ao mais endividado

PREVIEW_COLS = [
    'User_ID', 'savings_rate', 'pct_meses_negativo',
    'avg_saldo', 'spending_volatility',
    'top_category', 'top_category_pct',
    'has_investment', 'num_fixed_expenses',
    'avg_perc_gasto_credito', 'avg_variacao_gasto_mensal'
]

user_features[PREVIEW_COLS].sort_values('savings_rate', ascending=False).reset_index(drop=True)

,User_ID,savings_rate,pct_meses_negativo,avg_saldo,spending_volatility,top_category,top_category_pct,has_investment,num_fixed_expenses,avg_perc_gasto_credito,avg_variacao_gasto_mensal
0,USER_0049,0.633,1.0,-2510.888333,391.663060,Rent,0.351,1.0,5,6.270,23.839
1,USER_0014,0.632,1.0,-2371.357500,351.586007,Rent,0.380,1.0,5,4.829,15.574
2,USER_0015,0.609,1.0,-2315.831667,557.971474,Rent,0.381,1.0,5,5.669,40.766
3,USER_0044,0.606,1.0,-2368.543333,451.417868,Rent,0.350,1.0,5,4.334,7.739
4,USER_0050,0.577,1.0,-2395.552917,537.920610,Rent,0.372,1.0,5,6.954,15.134
5,USER_0009,0.563,1.0,-3083.285833,446.482043,Rent,0.295,1.0,5,4.128,8.610
6,USER_0039,0.563,1.0,-2486.012500,484.953506,Rent,0.342,1.0,5,4.127,6.882
7,USER_0021,0.525,1.0,-2344.672083,472.566364,Rent,0.381,1.0,5,4.409,0.423
8,USER_0042,0.525,1.0,-2361.311667,357.368284,Rent,0.348,1.0,5,7.575,34.999
9,USER_0045,0.463,1.0,-2442.816667,574.512663,Rent,0.345,1.0,5,8.820,23.167


## 4. Salvar

In [143]:
import os
os.makedirs('../data/processed', exist_ok=True)

user_features.to_csv('../data/processed/user_features.csv', index=False)
print('Features salvas em data/processed/user_features.csv')
print(f'{len(user_features)} usuarios x {user_features.shape[1]-1} features')
user_features.dtypes

Features salvas em data/processed/user_features.csv
51 usuarios x 19 features


User_ID                       object
avg_monthly_debit            float64
avg_monthly_credit           float64
avg_saldo                    float64
std_saldo                    float64
spending_volatility          float64
pct_meses_negativo           float64
total_meses                    int64
savings_rate                 float64
top_category                  object
top_category_spend           float64
total_debit                  float64
top_category_pct             float64
num_fixed_expenses             int64
total_investment             float64
has_investment               float64
avg_variacao_gasto_mensal    float64
std_variacao_gasto_mensal    float64
avg_perc_gasto_credito       float64
avg_qtd_transacoes_mes       float64
dtype: object